# Event Detection Cache — AMI Group 1 (Test Sequences)

Generates `detections_fusion.json` for the 5 FRED **test** sequences using:
- `fusion_event.pt` — Event component model (YOLOv8n, run2)

In [ ]:
!pip install -q ultralytics

In [ ]:
import re, json, shutil
from pathlib import Path

# Find fusion_event.pt — its parent is the dataset root
matches = list(Path('/kaggle/input').rglob('fusion_event.pt'))
if not matches:
    raise FileNotFoundError('fusion_event.pt not found — dataset not attached?')
DATASET = matches[0].parent
print('Dataset root:', DATASET)

# Show full structure (2 levels)
for p in sorted(DATASET.iterdir()):
    print(' ', p.name, '/' if p.is_dir() else f'  ({p.stat().st_size//1024} KB)')
    if p.is_dir():
        for q in sorted(p.iterdir())[:5]:
            print('   ', q.name, '/' if q.is_dir() else '')

WORK      = Path('/kaggle/working')
SEQUENCES = [8, 9, 12, 20, 21]

In [ ]:
from ultralytics import YOLO

event_model = YOLO(str(DATASET / 'fusion_event.pt'))
print('Event model loaded. nc:', event_model.model.nc, 'names:', event_model.names)

In [ ]:
EVENT_THRESHOLD = 0.1

def _numeric_key(p):
    m = re.search(r'_(\d+)\.png$', p.name)
    return int(m.group(1)) if m else 0

def _boxes(result):
    return [{'box': b.xyxy[0].tolist(), 'confidence': float(b.conf[0])}
            for b in result.boxes]

def find_event_dir(seq_n):
    """Find Event/Frames dir — handles both zip-extracted and direct dataset layouts."""
    # Kaggle auto-extracts zips: dataset_root/N/N/Event/Frames or dataset_root/N/Event/Frames
    candidates = [
        DATASET / str(seq_n) / str(seq_n) / 'Event' / 'Frames',
        DATASET / str(seq_n) / 'Event' / 'Frames',
        DATASET / str(seq_n) / 'Event' / 'images',
    ]
    for c in candidates:
        if c.exists() and any(c.iterdir()):
            return c
    return None

In [ ]:
BATCH = 32

for seq_n in SEQUENCES:
    seq_id   = f'sequence_{seq_n}'
    out_path = WORK / seq_id / 'detections_fusion.json'
    print(f'\n=== {seq_id} ===')

    event_dir = find_event_dir(seq_n)
    if event_dir is None:
        print(f'  WARNING: no event frames found — skipping')
        continue
    print(f'  Event dir: {event_dir}')

    event_files = sorted(event_dir.glob('*.png'), key=_numeric_key)
    n_frames = len(event_files)
    print(f'  Event frames: {n_frames}')
    if n_frames == 0:
        print('  WARNING: no PNG files — skipping')
        continue

    detections = []
    for i in range(0, n_frames, BATCH):
        b_evt = [str(f) for f in event_files[i:i+BATCH]]
        evt_results = event_model(b_evt, verbose=False, conf=0.01)

        for j, evt_r in enumerate(evt_results):
            frame_idx = i + j
            for det in _boxes(evt_r):
                if det['confidence'] >= EVENT_THRESHOLD:
                    x1, y1, x2, y2 = det['box']
                    detections.append({
                        'frame':      frame_idx,
                        'bbox':       [x1, y1, x2 - x1, y2 - y1],
                        'confidence': det['confidence'],
                        'class':      'drone',
                        'source':     'event',
                    })

        if (i // BATCH) % 10 == 0 or i + BATCH >= n_frames:
            print(f'  {min(i+BATCH, n_frames)}/{n_frames} frames  ({len(detections)} detections so far)')

    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps({
        'sequence_id': seq_id,
        'model':       'fusion',
        'cached':      True,
        'source_models': ['fusion_event_run2'],
        'detections':  detections,
    }, indent=2))
    print(f'  → {out_path}  ({len(detections)} detections)')

print('\n=== All sequences done ===')

In [ ]:
for seq_n in SEQUENCES:
    p = WORK / f'sequence_{seq_n}' / 'detections_fusion.json'
    if p.exists():
        d = json.loads(p.read_text())
        dets = d['detections']
        frames_with_dets = len(set(det['frame'] for det in dets))
        print(f'sequence_{seq_n}: {len(dets):5d} detections across {frames_with_dets} frames')
    else:
        print(f'sequence_{seq_n}: MISSING')

In [ ]:
import zipfile
zip_path = WORK / 'detections_fusion_test.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for seq_n in SEQUENCES:
        p = WORK / f'sequence_{seq_n}' / 'detections_fusion.json'
        if p.exists():
            zf.write(p, f'sequence_{seq_n}/detections_fusion.json')
            print(f'  added sequence_{seq_n}/detections_fusion.json')
print(f'Archive: {zip_path}  ({zip_path.stat().st_size//1024} KB)')